# Sales Prediction Model - Gold Layer

Trains a regression model to predict annual sales for expansion candidates using existing store data as training examples.

**Approach:**
- **Training data:** MI, VA, NY, WA, MD, NJ stores (~560 stores)
- **Validation:** Leave-One-Region-Out (LORO) cross-validation across 4 geographic folds
- **Final test:** MA hold-out (target expansion market)
- **Model:** XGBoost with constraints to prevent overfitting
- **Features:** Top 8 selected via RFE from demographics, POI counts, activity metrics, and trade area size
- **Target transformation:** Log-scaled (log1p) to handle sales variance

**Inputs:**
- `{catalog}.{gold_schema}.current_stores_features_agg` - Existing stores with sales data (training)
- `{catalog}.{gold_schema}.candidates_features_agg` - Candidate locations with aggregated features

**Outputs:**
- `{catalog}.{gold_schema}.candidates_finalized` - Ranked candidates with predicted sales
- `{catalog}.{gold_schema}.sales_prediction_model` - Registered MLflow model

## Setup

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

# Parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")
dbutils.widgets.text("min_predicted_sales", "250000")
dbutils.widgets.text("min_population", "5000")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
min_predicted_sales = int(dbutils.widgets.get("min_predicted_sales"))
min_population = int(dbutils.widgets.get("min_population"))

# Table names
training_table = f"{catalog}.{gold_schema}.current_stores_features_agg"
candidates_table = f"{catalog}.{gold_schema}.candidates_features_agg"
output_table = f"{catalog}.{gold_schema}.candidates_finalized"

print(f"Training data: {training_table}")
print(f"Candidates: {candidates_table}")
print(f"Output: {output_table}")
print(f"Min predicted sales: ${min_predicted_sales:,}")
print(f"Min population: {min_population:,}")

# Spatial CV configuration
TRAIN_STATES = ['MI', 'VA', 'NY', 'WA', 'MD', 'NJ']  # Training states (~560 stores)
TEST_STATE = 'MA'  # Hold-out state (target expansion market)

print(f"\nSpatial Cross-Validation:")
print(f"  Train states: {TRAIN_STATES}")
print(f"  Test state (hold-out): {TEST_STATE}")

# Set MLFlow experiment
mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/geospatial-retail-sales-prediction")
print(f"\n✓ MLFlow experiment configured")

## 1. Load and Prepare Data

In [ ]:
# Feature set: demographics, POI categories, activity index, and trade area size
# Note: total_poi_count removed (redundant with individual POI categories)
# Note: state_median_income removed (state-level constant = information leakage)
feature_columns = [
    # Demographics
    'population',
    'target_demographic_total',
    
    # POI counts by category (individual, not total)
    'retail', 'food_drink', 'leisure', 'education',
    'healthcare', 'financial', 'tourism', 'transportation',
    
    # Activity indicator
    'human_activity_index',
    
    # Trade area size (proxy for urbanity/density)
    'h3_cell_count',
]

target_column = 'annual_sales'

print(f"Features ({len(feature_columns)}): {feature_columns}")
print(f"Target: {target_column}")

In [ ]:
# Load training data
training_df = spark.table(training_table)
print(f"Loaded {training_df.count()} existing stores for training")

print("\nStore distribution by state:")
display(training_df.groupBy("state").count().orderBy("state"))

# Convert to Pandas
train_pd = training_df.select(feature_columns + [target_column, 'store_number', 'city', 'state']).toPandas()

# Normalize state values to abbreviations
state_mapping = {
    'Massachusetts': 'MA', 'massachusetts': 'MA',
    'Connecticut': 'CT', 'connecticut': 'CT',
    'New Jersey': 'NJ', 'new jersey': 'NJ',
    'Maryland': 'MD', 'maryland': 'MD',
    'Michigan': 'MI', 'michigan': 'MI',
    'Virginia': 'VA', 'virginia': 'VA',
    'New York': 'NY', 'new york': 'NY',
    'Washington': 'WA', 'washington': 'WA',
    'MA': 'MA', 'CT': 'CT', 'NJ': 'NJ', 'MD': 'MD',
    'MI': 'MI', 'VA': 'VA', 'NY': 'NY', 'WA': 'WA',
}
train_pd['state'] = train_pd['state'].map(lambda x: state_mapping.get(x, x))

print(f"\nState distribution after normalization:")
print(train_pd['state'].value_counts())

train_encoded = train_pd.copy()

print(f"\nData shape: {train_encoded.shape}")
print(f"Sample:")
display(train_encoded.head())

## 2. Correlation Analysis

In [ ]:
# Prepare features (numeric only)
X_initial = train_encoded.drop(columns=[target_column, 'store_number', 'city', 'state'])
X_initial = X_initial.select_dtypes(include=['number'])
y = train_encoded[target_column]

print(f"Initial feature set: {list(X_initial.columns)}")
print(f"Total features: {len(X_initial.columns)}")

# Correlation matrix
correlation_matrix = X_initial.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Find highly correlated pairs (|r| > 0.85)
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.85:
            high_corr.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr:
    print(f"\nHighly correlated pairs (|r| > 0.85): {len(high_corr)}")
    for feat1, feat2, corr in high_corr[:10]:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("\n✓ No high multicollinearity detected")

# Correlation with target
target_corr = X_initial.corrwith(y).sort_values(ascending=False)
print(f"\nTop features correlated with {target_column}:")
print(target_corr)

## 3. Feature Selection (RFE)

In [ ]:
# Recursive Feature Elimination to select top 8 features
# With ~560 training stores, we can support more features than with 42
rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
n_features_to_select = min(8, len(X_initial.columns) - 1)

rfe = RFE(estimator=rf_estimator, n_features_to_select=n_features_to_select)
rfe.fit(X_initial, y)

rfe_selected = X_initial.columns[rfe.support_]
print(f"RFE selected {len(rfe_selected)} features:")
print(list(rfe_selected))

# Feature ranking (1 = selected, higher = eliminated earlier)
feature_ranking = pd.DataFrame({
    'feature': X_initial.columns,
    'ranking': rfe.ranking_
}).sort_values('ranking')

print(f"\nFeature ranking:")
print(feature_ranking)

In [ ]:
# Use only RFE selected features
final_features = list(rfe_selected)
X_final = X_initial[final_features]

print(f"\n{'='*60}")
print(f"FINAL FEATURE SET: {len(final_features)} features")
print(f"{'='*60}")
for i, feat in enumerate(sorted(final_features), 1):
    print(f"{i:2d}. {feat}")

print(f"\n✓ Reduced from {len(X_initial.columns)} to {len(final_features)} features")

In [ ]:
# Spatial cross-validation: train on MI, VA, NY, WA, MD, NJ → test on MA
train_mask = train_encoded['state'].isin(TRAIN_STATES)
test_mask = train_encoded['state'] == TEST_STATE

train_count = train_mask.sum()
test_count = test_mask.sum()

print("=" * 60)
print("SPATIAL CROSS-VALIDATION SPLIT")
print("=" * 60)

# Validate split
if train_count == 0:
    raise ValueError(f"No training data! States in data: {train_encoded['state'].unique()}")
if test_count == 0:
    raise ValueError(f"No test data! States in data: {train_encoded['state'].unique()}")

print(f"\nTraining States: {TRAIN_STATES}")
print(f"  Stores: {train_count}")
for state in TRAIN_STATES:
    state_count = (train_encoded['state'] == state).sum()
    print(f"    {state}: {state_count} stores")

print(f"\nTest State (Hold-Out): {TEST_STATE}")
print(f"  Stores: {test_count}")

# Create train/test sets
X_train = X_final[train_mask]
X_test = X_final[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

# Apply log transformation to target
print(f"\n{'='*60}")
print("APPLYING LOG SCALING TO TARGET VARIABLE")
print("="*60)

print(f"\nOriginal sales distribution:")
print(f"  Min: ${y_train.min():,.0f}")
print(f"  Max: ${y_train.max():,.0f}")
print(f"  Mean: ${y_train.mean():,.0f}")
print(f"  Std: ${y_train.std():,.0f}")

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"\nLog-transformed distribution:")
print(f"  Min: {y_train_log.min():.3f}")
print(f"  Max: {y_train_log.max():.3f}")
print(f"  Mean: {y_train_log.mean():.3f}")
print(f"  Std: {y_train_log.std():.3f}")

print(f"\n{'='*60}")
print(f"Final Split:")
print(f"  Train: {len(X_train)} stores ({100*len(X_train)/(len(X_train)+len(X_test)):.0f}%)")
print(f"  Test:  {len(X_test)} stores ({100*len(X_test)/(len(X_train)+len(X_test)):.0f}%)")
print(f"  Target: log-scaled sales")
print(f"{'='*60}")

# Visualize the split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Split visualization
train_label = ', '.join(TRAIN_STATES)
split_data = pd.DataFrame({
    'Set': [f'Train ({train_label})', f'Test ({TEST_STATE})'],
    'Stores': [len(X_train), len(X_test)]
})
split_data.plot(x='Set', y='Stores', kind='bar', ax=axes[0], color=['steelblue', 'coral'], legend=False)
axes[0].set_ylabel('Number of Stores')
axes[0].set_title('Spatial Cross-Validation Split')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(split_data['Stores']):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Original vs Log-transformed distribution
axes[1].hist(y_train, bins=15, alpha=0.7, color='steelblue')
axes[1].set_xlabel('Annual Sales ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Original Sales Distribution')

axes[2].hist(y_train_log, bins=15, alpha=0.7, color='coral')
axes[2].set_xlabel('Log(1 + Annual Sales)')
axes[2].set_ylabel('Count')
axes[2].set_title('Log-Transformed Sales Distribution')

plt.tight_layout()
plt.show()

## 4.5 Leave-One-Region-Out (LORO) Cross-Validation

Evaluates model generalization across geographic regions before final training.
- 4 geographic folds based on regional proximity
- MA excluded from all folds (reserved for final validation)
- Reports mean and std of RMSE/R² for confidence intervals

In [ ]:
# Leave-One-Region-Out Cross-Validation
# Evaluates how well the model generalizes across different geographies
# MA is excluded from all folds - reserved for final validation only

REGIONAL_FOLDS = {
    'fold_great_lakes': {
        'test': ['MI'],
        'description': 'Hold out Great Lakes'
    },
    'fold_mid_atlantic': {
        'test': ['VA', 'MD'],
        'description': 'Hold out Mid-Atlantic'
    },
    'fold_northeast': {
        'test': ['NY', 'NJ'],
        'description': 'Hold out Northeast Corridor'
    },
    'fold_pacific': {
        'test': ['WA'],
        'description': 'Hold out Pacific Northwest'
    },
}

print("=" * 60)
print("LEAVE-ONE-REGION-OUT CROSS-VALIDATION")
print("=" * 60)

loro_results = []

for fold_name, fold_config in REGIONAL_FOLDS.items():
    # Train on all non-MA training states except the test fold
    train_states_fold = [s for s in TRAIN_STATES if s not in fold_config['test']]
    test_states_fold = fold_config['test']
    
    fold_train_mask = train_encoded['state'].isin(train_states_fold)
    fold_test_mask = train_encoded['state'].isin(test_states_fold)
    
    X_train_fold = X_final[fold_train_mask]
    X_test_fold = X_final[fold_test_mask]
    y_train_fold = y[fold_train_mask]
    y_test_fold = y[fold_test_mask]
    
    if len(X_train_fold) == 0 or len(X_test_fold) == 0:
        print(f"  Skipping {fold_name}: insufficient data")
        continue
    
    # Train XGBoost on this fold
    fold_model = xgb.XGBRegressor(
        n_estimators=100, max_depth=3, min_child_weight=5,
        learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
    )
    fold_model.fit(X_train_fold, np.log1p(y_train_fold))
    
    # Evaluate in original dollar scale
    y_pred_fold = np.expm1(fold_model.predict(X_test_fold))
    fold_rmse = np.sqrt(mean_squared_error(y_test_fold, y_pred_fold))
    fold_r2 = r2_score(y_test_fold, y_pred_fold)
    fold_mae = mean_absolute_error(y_test_fold, y_pred_fold)
    
    loro_results.append({
        'fold': fold_name,
        'test_states': ', '.join(test_states_fold),
        'train_size': len(X_train_fold),
        'test_size': len(X_test_fold),
        'rmse': fold_rmse,
        'r2': fold_r2,
        'mae': fold_mae
    })
    
    print(f"\n{fold_name} ({fold_config['description']})")
    print(f"  Train: {len(X_train_fold)} stores | Test: {len(X_test_fold)} stores")
    print(f"  RMSE: ${fold_rmse:,.0f} | R²: {fold_r2:.3f} | MAE: ${fold_mae:,.0f}")

# Summary across all folds
loro_df = pd.DataFrame(loro_results)
print(f"\n{'='*60}")
print("LORO Cross-Validation Summary")
print(f"{'='*60}")
print(f"  Mean RMSE: ${loro_df['rmse'].mean():,.0f} (±${loro_df['rmse'].std():,.0f})")
print(f"  Mean R²:   {loro_df['r2'].mean():.3f} (±{loro_df['r2'].std():.3f})")
print(f"  Mean MAE:  ${loro_df['mae'].mean():,.0f}")

# Log LORO results to MLflow
with mlflow.start_run(run_name="loro_cross_validation"):
    mlflow.log_param("cv_strategy", "LORO")
    mlflow.log_param("n_folds", len(loro_results))
    mlflow.log_param("features", ",".join(final_features))
    
    for result in loro_results:
        mlflow.log_metric(f"rmse_{result['fold']}", result['rmse'])
        mlflow.log_metric(f"r2_{result['fold']}", result['r2'])
    
    mlflow.log_metric("mean_cv_rmse", loro_df['rmse'].mean())
    mlflow.log_metric("mean_cv_r2", loro_df['r2'].mean())
    mlflow.log_metric("std_cv_rmse", loro_df['rmse'].std())
    mlflow.log_metric("std_cv_r2", loro_df['r2'].std())

display(loro_df)

print(f"\n✓ LORO results logged to MLflow")

## 5. Model Training

### 5.1 Baseline: Linear Regression

In [ ]:
with mlflow.start_run(run_name="baseline_linear_regression_log_scaled"):
    # Train on log-scaled target
    baseline_model = LinearRegression()
    baseline_model.fit(X_train, y_train_log)
    
    # Predictions in log scale
    y_pred_train_log = baseline_model.predict(X_train)
    y_pred_test_log = baseline_model.predict(X_test)
    
    # Convert back to original scale
    y_pred_train = np.expm1(y_pred_train_log)
    y_pred_test = np.expm1(y_pred_test_log)
    
    # Metrics in original scale
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_r2_log = r2_score(y_test_log, y_pred_test_log)
    
    # Log to MLFlow
    mlflow.log_params({
        "model_type": "LinearRegression",
        "target_transform": "log1p",
        "n_features": len(final_features),
        "split_method": "spatial_holdout_state",
        "train_states": ",".join(TRAIN_STATES),
        "test_state": TEST_STATE,
        "train_size": len(X_train),
        "test_size": len(X_test)
    })
    
    mlflow.log_metrics({
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "test_r2_log_scale": test_r2_log
    })
    
    # Diagnostics plot
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    ax[0].scatter(y_pred_test, y_test - y_pred_test, alpha=0.6)
    ax[0].axhline(y=0, color='r', linestyle='--')
    ax[0].set_xlabel('Predicted Sales ($)')
    ax[0].set_ylabel('Residuals ($)')
    ax[0].set_title('Residual Plot (MA Hold-Out)')
    
    ax[1].scatter(y_test, y_pred_test, alpha=0.6)
    ax[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    ax[1].set_xlabel('Actual Sales ($)')
    ax[1].set_ylabel('Predicted Sales ($)')
    ax[1].set_title(f'Actual vs Predicted - MA (R²={test_r2:.3f})')
    
    plt.tight_layout()
    mlflow.log_figure(fig, "baseline_diagnostics.png")
    plt.show()
    
    mlflow.sklearn.log_model(baseline_model, "model")
    
    baseline_metrics = {"test_rmse": test_rmse, "test_r2": test_r2, "test_r2_log": test_r2_log}
    
    print(f"\nBaseline Linear Regression (Log-Scaled Target):")
    print(f"  Train RMSE: ${train_rmse:,.0f}")
    print(f"  Test RMSE (MA): ${test_rmse:,.0f}")
    print(f"  Test MAE (MA): ${test_mae:,.0f}")
    print(f"  Test R² (original scale): {test_r2:.3f}")
    print(f"  Test R² (log scale): {test_r2_log:.3f}")

### 5.2 Advanced: XGBoost

In [ ]:
with mlflow.start_run(run_name="xgboost_constrained_log_scaled"):
    # Train XGBoost with constrained hyperparameters to prevent overfitting
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=3,              # Simpler trees
        min_child_weight=5,       # More samples per leaf
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,            # L1 regularization
        reg_lambda=1.0,           # L2 regularization
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train_log)
    
    # Predictions in log scale
    y_pred_train_log = xgb_model.predict(X_train)
    y_pred_test_log = xgb_model.predict(X_test)
    
    # Convert back to original scale
    y_pred_train = np.expm1(y_pred_train_log)
    y_pred_test = np.expm1(y_pred_test_log)
    
    # Metrics in original scale
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_r2_log = r2_score(y_test_log, y_pred_test_log)
    
    # Log to MLFlow
    mlflow.log_params({
        "model_type": "XGBRegressor",
        "target_transform": "log1p",
        "n_estimators": 100,
        "max_depth": 3,
        "min_child_weight": 5,
        "learning_rate": 0.1,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_features": len(final_features),
        "split_method": "spatial_holdout_state",
        "train_states": ",".join(TRAIN_STATES),
        "test_state": TEST_STATE,
        "train_size": len(X_train),
        "test_size": len(X_test)
    })
    
    mlflow.log_metrics({
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "test_r2_log_scale": test_r2_log
    })
    
    # Feature importance
    fig, ax = plt.subplots(figsize=(10, 6))
    xgb.plot_importance(xgb_model, ax=ax, max_num_features=15, importance_type='gain')
    plt.title('XGBoost Feature Importance (Gain)')
    plt.tight_layout()
    mlflow.log_figure(fig, "xgb_feature_importance.png")
    plt.show()
    
    # Diagnostics
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    ax[0].scatter(y_pred_test, y_test - y_pred_test, alpha=0.6)
    ax[0].axhline(y=0, color='r', linestyle='--')
    ax[0].set_xlabel('Predicted Sales ($)')
    ax[0].set_ylabel('Residuals ($)')
    ax[0].set_title('Residual Plot (MA Hold-Out)')
    
    ax[1].scatter(y_test, y_pred_test, alpha=0.6)
    ax[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    ax[1].set_xlabel('Actual Sales ($)')
    ax[1].set_ylabel('Predicted Sales ($)')
    ax[1].set_title(f'Actual vs Predicted - MA (R²={test_r2:.3f})')
    
    plt.tight_layout()
    mlflow.log_figure(fig, "xgb_diagnostics.png")
    plt.show()
    
    mlflow.sklearn.log_model(xgb_model, "model")
    
    xgb_metrics = {"test_rmse": test_rmse, "test_r2": test_r2, "test_r2_log": test_r2_log}
    
    print(f"\nXGBoost (Constrained, Log-Scaled Target):")
    print(f"  Hyperparameters: max_depth=3, min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0")
    print(f"  Train RMSE: ${train_rmse:,.0f}")
    print(f"  Test RMSE (MA): ${test_rmse:,.0f}")
    print(f"  Test MAE (MA): ${test_mae:,.0f}")
    print(f"  Test R² (original scale): {test_r2:.3f}")
    print(f"  Test R² (log scale): {test_r2_log:.3f}")

## 6. SHAP Analysis (Explainability)

In [ ]:
# Create SHAP explainer for XGBoost
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print("SHAP values computed for test set")
print(f"Shape: {shap_values.shape}")

In [ ]:
# SHAP Summary Plot (global feature importance)
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Sales Predictions')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Features listed by importance (top to bottom)")
print("- Red = high feature value, Blue = low feature value")
print("- Position on x-axis shows impact on prediction")

In [ ]:
# SHAP Feature Importance Bar Chart
shap_importance = pd.DataFrame({
    'feature': X_test.columns,
    'mean_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False)

print("Top 10 Features by SHAP Importance:")
print(shap_importance.head(10))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
shap_importance.head(10).plot(x='feature', y='mean_shap', kind='barh', ax=ax, color='steelblue')
plt.xlabel('Mean |SHAP value|')
plt.ylabel('Feature')
plt.title('Top 10 Features by SHAP Importance')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Waterfall Plot (explain single prediction)
sample_idx = 0
fig, ax = plt.subplots(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value,
        data=X_test.iloc[sample_idx],
        feature_names=X_test.columns.tolist()
    ),
    show=False
)
plt.title(f'SHAP Waterfall: How features contribute to prediction for store #{sample_idx}')
plt.tight_layout()
plt.show()

print(f"\nActual sales: ${y_test.iloc[sample_idx]:,.0f}")
print(f"Predicted sales: ${y_pred_test[sample_idx]:,.0f}")

## 7. Model Selection & Registration

In [ ]:
# Compare models
print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"\nBaseline (Linear Regression):")
print(f"  RMSE: ${baseline_metrics['test_rmse']:,.0f}")
print(f"  R²: {baseline_metrics['test_r2']:.3f}")

print(f"\nXGBoost:")
print(f"  RMSE: ${xgb_metrics['test_rmse']:,.0f}")
print(f"  R²: {xgb_metrics['test_r2']:.3f}")

# Select best model (lower RMSE = better)
if xgb_metrics['test_rmse'] < baseline_metrics['test_rmse']:
    best_model = xgb_model
    best_model_name = "XGBoost"
    best_rmse = xgb_metrics['test_rmse']
else:
    best_model = baseline_model
    best_model_name = "LinearRegression"
    best_rmse = baseline_metrics['test_rmse']

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*60}")

In [ ]:
# Register to Unity Catalog
model_name = f"{catalog}.{gold_schema}.sales_prediction_model"

with mlflow.start_run(run_name=f"PRODUCTION_{best_model_name}"):
    mlflow.log_params({
        "model_type": best_model_name,
        "features": final_features
    })
    
    model_info = mlflow.sklearn.log_model(
        best_model,
        "model",
        registered_model_name=model_name,
        input_example=X_test.iloc[:5],
        signature=mlflow.models.infer_signature(X_test, y_test)
    )

print(f"\n✓ Model registered to Unity Catalog:")
print(f"  Name: {model_name}")
print(f"  Type: {best_model_name}")

## 8. Predict for Candidates

In [ ]:
# Load candidates with aggregated features
candidates_df = spark.table(candidates_table)
print(f"Loaded {candidates_df.count():,} expansion candidates")

print("\nCandidates by state:")
display(candidates_df.groupBy("state").count().orderBy("state"))

# Convert to Pandas and prepare features
candidates_pd = candidates_df.select(['candidate_id'] + feature_columns + ['state']).toPandas()

# Normalize state values
state_mapping = {
    'Massachusetts': 'MA', 'massachusetts': 'MA',
    'Connecticut': 'CT', 'connecticut': 'CT',
    'New Jersey': 'NJ', 'new jersey': 'NJ',
    'Maryland': 'MD', 'maryland': 'MD',
    'Michigan': 'MI', 'michigan': 'MI',
    'Virginia': 'VA', 'virginia': 'VA',
    'New York': 'NY', 'new york': 'NY',
    'Washington': 'WA', 'washington': 'WA',
    'MA': 'MA', 'CT': 'CT', 'NJ': 'NJ', 'MD': 'MD',
    'MI': 'MI', 'VA': 'VA', 'NY': 'NY', 'WA': 'WA',
}
candidates_pd['state'] = candidates_pd['state'].map(lambda x: state_mapping.get(x, x))

candidates_encoded = candidates_pd.copy()

# Align columns with training data
missing_cols = set(final_features) - set(candidates_encoded.columns)
for col in missing_cols:
    candidates_encoded[col] = 0
    print(f"Warning: Missing column '{col}' - filled with 0")
    
X_candidates = candidates_encoded[['candidate_id'] + final_features]

print(f"\nPrepared {len(X_candidates)} candidates for prediction")
print(f"Using features: {final_features}")

In [ ]:
# Predict sales for candidates
X_pred = X_candidates.drop(columns=['candidate_id'])

# Predictions in log scale
predictions_log = best_model.predict(X_pred)

# Convert back to original scale
predictions = np.expm1(predictions_log)
predictions = np.maximum(predictions, 0)  # Floor at 0

# Create results DataFrame
results_pd = pd.DataFrame({
    'candidate_id': X_candidates['candidate_id'],
    'predicted_annual_sales': predictions.astype(int),
    'predicted_log_sales': predictions_log,
    'prediction_interval_lower': np.maximum(0, (predictions - best_rmse)).astype(int),
    'prediction_interval_upper': (predictions + best_rmse).astype(int),
    'model_version': f"{best_model_name}_log_scaled"
})

print(f"\nPrediction Summary:")
print(f"  Count: {len(results_pd):,}")
print(f"  Mean: ${results_pd['predicted_annual_sales'].mean():,.0f}")
print(f"  Median: ${results_pd['predicted_annual_sales'].median():,.0f}")
print(f"  Min: ${results_pd['predicted_annual_sales'].min():,.0f}")
print(f"  Max: ${results_pd['predicted_annual_sales'].max():,.0f}")

display(results_pd.head(10))

## 9. Apply Business Constraints, Rank & Write Output

In [ ]:
# Convert predictions to Spark and join back to candidates
predictions_spark = spark.createDataFrame(results_pd)

candidates_with_predictions = candidates_df.join(
    predictions_spark,
    "candidate_id",
    "inner"
).withColumn(
    "predicted_monthly_sales",
    (col("predicted_annual_sales") / 12).cast("long")
)

print(f"Joined predictions to {candidates_with_predictions.count():,} candidates")

# Apply business constraints (currently disabled)
candidates_constrained = candidates_with_predictions

before_count = candidates_with_predictions.count()
after_count = candidates_constrained.count()
print(f"\nBusiness constraints:")
print(f"  Before: {before_count:,}")
print(f"  After: {after_count:,}")
print(f"  Filtered: {before_count - after_count:,}")

# Rank candidates by predicted sales
window_spec = Window.orderBy(F.desc("predicted_annual_sales"))
candidates_ranked = candidates_constrained.withColumn(
    "rank", F.row_number().over(window_spec)
).withColumn(
    "percentile", F.percent_rank().over(window_spec)
)

# Add metadata
candidates_final = candidates_ranked.withColumn(
    "prediction_timestamp", F.current_timestamp()
).withColumn(
    "min_sales_threshold", F.lit(min_predicted_sales)
).withColumn(
    "min_population_threshold", F.lit(min_population)
)

# Write to gold
(
    candidates_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written {candidates_final.count():,} ranked candidates to {output_table}")

# Show top candidates
print("\nTop 20 Expansion Candidates:")
display(candidates_final.select(
    "rank", "candidate_id", "latitude", "longitude", "state", "urbanity",
    "population", "target_demographic_total", "total_poi_count",
    "predicted_annual_sales", "predicted_monthly_sales"
).limit(20))

## Summary

In [ ]:
print("="*60)
print("SALES PREDICTION MODEL - SUMMARY")
print("="*60)

print(f"\n1. Training Data:")
print(f"   - {len(train_encoded)} stores total")
print(f"   - Train: {TRAIN_STATES} ({len(X_train)} stores)")
print(f"   - Test: {TEST_STATE} ({len(X_test)} stores)")

print(f"\n2. Features:")
print(f"   - Selected: {len(final_features)} features via RFE")
print(f"   - {final_features}")

print(f"\n3. LORO Cross-Validation:")
print(f"   - Mean RMSE: ${loro_df['rmse'].mean():,.0f} (±${loro_df['rmse'].std():,.0f})")
print(f"   - Mean R²: {loro_df['r2'].mean():.3f}")

print(f"\n4. Best Model (MA Hold-Out):")
print(f"   - Type: {best_model_name}")
print(f"   - Test RMSE (MA): ${best_rmse:,.0f}")
print(f"   - Test R²: {xgb_metrics['test_r2']:.3f}")
print(f"   - Registered: {model_name}")

print(f"\n5. Predictions:")
print(f"   - Total candidates: {len(results_pd):,}")
print(f"   - After constraints: {candidates_final.count():,}")
print(f"   - Output: {output_table}")

print(f"\n{'='*60}")
print("✓ COMPLETE")
print("="*60)